In [21]:
# Importing necessary modules, with auto-install for missing packages.

import subprocess, sys
 
# Auto-install any missing packages (safe to re-run)
_required = ["requests", "beautifulsoup4", "lxml", "pandas", "matplotlib", "pillow"]
for _pkg in _required:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", _pkg, "-q"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
 
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import os
import re
import time
import csv
from datetime import datetime
import matplotlib
matplotlib.use("Agg")   # non-interactive backend -- works in any Jupyter
import matplotlib.pyplot as plt
import matplotlib.animation as animation
 
print("All modules installed and imported successfully!")
print(f"  Python   : {sys.version.split()[0]}")
print(f"  Pandas   : {pd.__version__}")
print(f"  Requests : {requests.__version__}")
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SHARED HELPER — scrape_weather()
# Used by both Exercise 1 and Exercise 2.
# Source: timeanddate.com/weather/philippines/<city_slug>
# ─────────────────────────────────────────────────────────────────────────────
 
_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}
 
 
def _clean(text):
    """Remove non-breaking spaces and collapse whitespace."""
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()
 
 
def _table_value(soup, keyword):
    """
    Walk every table row; return the adjacent cell's text
    when the first cell contains keyword.
    """
    for row in soup.find_all("tr"):
        cells = row.find_all(["td", "th"])
        for i, cell in enumerate(cells):
            if keyword.lower() in cell.get_text().lower() and i + 1 < len(cells):
                return _clean(cells[i + 1].get_text())
    return "N/A"
 
 
def scrape_weather(city_slug):
    """
    Scrape current + extended forecast weather from timeanddate.com
    for a Philippine city.
 
    city_slug : URL slug, e.g. 'manila', 'cebu', 'davao'.
 
    Returns a dict with:
        temperature, feels_like, condition, temp_high, temp_low,
        wind, humidity, pressure, visibility, dew_point,
        forecast_rows (list of dicts: forecast_date, temp_hi, temp_lo,
                       condition, humidity)
    Returns None on network failure.
    """
    base_url     = f"https://www.timeanddate.com/weather/philippines/{city_slug}"
    forecast_url = f"{base_url}/ext"
 
    try:
        resp = requests.get(base_url, headers=_HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.exceptions.RequestException as exc:
        print(f"  Could not reach {base_url}: {exc}")
        return None
 
    soup = BeautifulSoup(resp.text, "lxml")
 
    # Current temperature
    temp_el = soup.select_one("#qlook .h2") or soup.select_one(".h2")
    temperature = _clean(temp_el.get_text()) if temp_el else "N/A"
 
    # Weather condition
    cond_el = soup.select_one("#qlook p")
    condition = _clean(cond_el.get_text()) if cond_el else _table_value(soup, "Condition")
 
    # Forecast high / low
    hi_el = soup.select_one(".hi-lo .hi")
    lo_el = soup.select_one(".hi-lo .lo")
    temp_high = _clean(hi_el.get_text()) if hi_el else "N/A"
    temp_low  = _clean(lo_el.get_text()) if lo_el else "N/A"
 
    # Detailed attributes
    feels_like = _table_value(soup, "Feels Like")
    wind       = _table_value(soup, "Wind")
    humidity   = _table_value(soup, "Humidity")
    pressure   = _table_value(soup, "Pressure")
    visibility = _table_value(soup, "Visibility")
    dew_point  = _table_value(soup, "Dew Point")
 
    # Extended forecast page
    forecast_rows = []
    try:
        freq = requests.get(forecast_url, headers=_HEADERS, timeout=15)
        freq.raise_for_status()
        fsoup = BeautifulSoup(freq.text, "lxml")
        tbl = fsoup.select_one("#wt-ext") or fsoup.find("table")
        if tbl:
            for row in tbl.find_all("tr")[1:]:
                cols = row.find_all("td")
                if len(cols) >= 3:
                    forecast_rows.append({
                        "forecast_date": _clean(cols[0].get_text()),
                        "temp_hi":       _clean(cols[1].get_text()),
                        "temp_lo":       _clean(cols[2].get_text()),
                        "condition":     _clean(cols[3].get_text()) if len(cols) > 3 else "N/A",
                        "humidity":      _clean(cols[5].get_text()) if len(cols) > 5 else "N/A",
                    })
    except requests.exceptions.RequestException:
        pass   # forecast is optional
 
    return {
        "temperature":   temperature,
        "feels_like":    feels_like,
        "condition":     condition,
        "temp_high":     temp_high,
        "temp_low":      temp_low,
        "wind":          wind,
        "humidity":      humidity,
        "pressure":      pressure,
        "visibility":    visibility,
        "dew_point":     dew_point,
        "forecast_rows": forecast_rows,
    }

All modules installed and imported successfully!
  Python   : 3.14.2
  Pandas   : 3.0.2
  Requests : 2.33.1


In [1]:
# 1. Daily Weather Logger Simulation (Final Stable Version)

import requests
import csv
import time
from datetime import datetime

print("=== Daily Weather Logger ===")

city = input("Enter Philippine city (Manila, Cebu, Davao): ").strip().lower()

coordinates = {
    "manila": (14.5995, 120.9842),
    "cebu": (10.3157, 123.8854),
    "davao": (7.1907, 125.4553)
}

if city not in coordinates:
    print("City not supported.")
    exit()

try:
    days = int(input("Enter number of simulated runs (default 3): "))
    if days <= 0:
        days = 3
except:
    days = 3

lat, lon = coordinates[city]

filename = "weather_data.csv"

for i in range(days):
    print(f"\nCollecting data run {i+1}...")

    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"

    response = requests.get(url)
    data = response.json()

    temperature = data["current_weather"]["temperature"]

    with open(filename, "a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow([datetime.now(), city, f"{temperature}°C"])

    time.sleep(2)

print("\nWeather logging completed.")

=== Daily Weather Logger ===




Weather logging completed.


In [2]:
# 2. JSON Weather Exporter (Final Stable Version)

import requests
import json
from datetime import datetime

print("=== JSON Weather Exporter ===")

city = input("Enter Philippine city (Manila, Cebu, Davao): ").strip().lower()

coordinates = {
    "manila": (14.5995, 120.9842),
    "cebu": (10.3157, 123.8854),
    "davao": (7.1907, 125.4553)
}

if city not in coordinates:
    print("City not supported.")
    exit()

lat, lon = coordinates[city]

url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"

response = requests.get(url)
data = response.json()

record = {
    "timestamp": str(datetime.now()),
    "city": city,
    "temperature": data["current_weather"]["temperature"]
}

filename = "weather_data.json"

try:
    with open(filename, "r", encoding="utf-8") as f:
        weather_list = json.load(f)
except:
    weather_list = []

weather_list.append(record)

with open(filename, "w", encoding="utf-8") as f:
    json.dump(weather_list, f, indent=4)

print("Weather data saved to JSON.")

=== JSON Weather Exporter ===
Weather data saved to JSON.


In [4]:
# Forecast Data Visualizer

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation

print("=== Forecast Data Visualizer ===")

city = input("Enter city to visualize: ").strip().lower()

try:
    records = int(input("Number of recent records to visualize (default 3): "))
except:
    records = 3

#  Read CSV safely (Windows encoding)
df = pd.read_csv("weather_data.csv", encoding="cp1252")

#  If file has no headers, assign them manually
if len(df.columns) == 3:
    df.columns = ["timestamp", "city", "temperature"]

elif len(df.columns) == 4:
    # If extra empty column exists, drop it
    df = df.iloc[:, :3]
    df.columns = ["timestamp", "city", "temperature"]

#  Filter selected city
df_city = df[df["city"].str.lower() == city].tail(records)

if df_city.empty:
    print("No data found for this city.")
    exit()

temps = []

for t in df_city["temperature"]:
    try:
        value = float(str(t).replace("°C", "").strip())
        temps.append(value)
    except:
        print(f"Skipping invalid temperature value: {t}")

if not temps:
    print("No valid temperature data found.")
else:
    fig, ax = plt.subplots()

    def update(frame):
        ax.clear()
        ax.plot(temps[:frame+1], marker="o")
        ax.set_title(f"Temperature Trend - {city.title()}")
        ax.set_xlabel("Record Number")
        ax.set_ylabel("Temperature (°C)")
        ax.grid(True)

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(temps),
        repeat=False
    )

    ani.save("forecast_plot.gif", writer="pillow")
    print("Animation saved as forecast_plot.gif")

    plt.show()

=== Forecast Data Visualizer ===
Skipping invalid temperature value: 33.3Â°C
Skipping invalid temperature value: 33.3Â°C
No valid temperature data found.
